# 🧠 Brain Tumor Detection - YOLOv8 Training in Google Colab

This notebook will guide you through training your brain tumor detection model in Google Colab.

## 📋 Setup Instructions:
1. Upload your project files to Google Drive
2. Run each cell in order
3. Follow the prompts when they appear

---


## Step 1: Connect to Google Drive

This connects your Colab notebook to your Google Drive so you can access your files.


In [ ]:
# ==================== STEP 1: CONNECT TO GOOGLE DRIVE ====================
# This code connects your Colab notebook to your Google Drive
# so you can access your files

from google.colab import drive
# This function mounts (connects) your Google Drive to Colab
# The parameter '/content/drive' tells Colab where to put the Drive files
drive.mount('/content/drive')


## Step 2: Navigate to Your Project Folder

Change to your project directory. **IMPORTANT:** Replace `'BrainTumorYolov8'` with your actual folder name if it's different!


In [ ]:
# ==================== STEP 2: GO TO YOUR PROJECT FOLDER ====================
# This changes your current directory to your project folder
# Replace 'BrainTumorYolov8' with your actual folder name if different

import os
# os.chdir() changes the current working directory
# '/content/drive/MyDrive' is the default path to your Google Drive
# 'BrainTumorYolov8' is the name of your project folder
os.chdir('/content/drive/MyDrive/BrainTumorYolov8')

# Print the current directory to verify
# os.getcwd() gets the current working directory
print("Current directory:", os.getcwd())

# List all files in this directory
# os.listdir() lists all files and folders in the current directory
print("\nFiles in this folder:")
for file in os.listdir('.'):
    print(f"  - {file}")


## Step 3: Extract Dataset (If You Uploaded a Zip File)

If you uploaded your dataset as individual folders, **skip this cell**. Only run this if you uploaded `brain_tumor_dataset.zip`.


In [ ]:
# ==================== STEP 3: EXTRACT DATASET ====================
# This code extracts (unzips) your dataset if you uploaded it as a zip file
# Skip this step if you uploaded individual folders

import zipfile
# zipfile is a Python library for working with zip files

# Check if the zip file exists
zip_file_path = 'brain_tumor_dataset.zip'
# os.path.exists() checks if a file or folder exists at the given path
if os.path.exists(zip_file_path):
    print("Found zip file! Extracting...")
    
    # zipfile.ZipFile() opens the zip file for reading
    # The parameter is the path to the zip file
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        # zip_ref.extractall() extracts all files from the zip
        # The parameter '.' means extract to the current directory
        zip_ref.extractall('.')
    
    print("Extraction complete!")
else:
    print("Zip file not found. If you uploaded individual folders, skip this step.")


## Step 3b: Fix Nested Dataset Structure (If Needed)

If your dataset is in a nested `brain_tumor_dataset/` folder, this cell moves it to the expected location.



In [ ]:
# ==================== STEP 3B: FIX NESTED DATASET STRUCTURE ====================
# If your dataset is in a nested 'brain_tumor_dataset/' folder, 
# this moves train/valid/test to the expected location

import os
import shutil

nested_dataset = 'brain_tumor_dataset'

if os.path.exists(nested_dataset):
    print(f"Found nested folder: {nested_dataset}")
    print("Moving train/, valid/, test/ to current directory...")
    
    # Copy each folder from nested location to current directory
    for folder_name in ['train', 'valid', 'test']:
        src = os.path.join(nested_dataset, folder_name)
        dst = os.path.join('.', folder_name)
        
        if os.path.exists(src):
            if os.path.exists(dst):
                print(f"  {folder_name}/ already exists, skipping")
            else:
                shutil.copytree(src, dst)
                print(f"  ✅ Copied {src} → {dst}")
        else:
            print(f"  ⚠️ {src} not found")
    
    print("\nDataset structure fixed!")
else:
    print(f"No nested '{nested_dataset}' folder found.")
    print("Your dataset is already in the expected structure.")



## Step 4: Install Required Python Libraries

This installs all the packages your project needs. This may take a few minutes.


In [ ]:
# ==================== STEP 4: INSTALL PYTHON LIBRARIES ====================
# This code installs all the Python packages your project needs
# pip is a package manager for Python (like an app store for code libraries)

# Install ultralytics (YOLOv8 library)
# The exclamation mark (!) tells Colab to run this as a shell command
!pip install ultralytics

# Install streamlit (for the web app, optional)
!pip install streamlit

# Install other required libraries
!pip install pandas
!pip install plotly
!pip install Pillow

# Verify installation
print("All packages installed successfully!")


## Step 5: Verify Dataset Structure

This checks if your dataset folders are set up correctly before training.


In [ ]:
# ==================== STEP 5: CHECK DATASET STRUCTURE ====================
# This code checks if your dataset folders are set up correctly
# It's important to verify before training

# Define the expected folder structure
# These are the folders YOLO expects to find
expected_folders = ['train/images', 'train/labels', 
                    'valid/images', 'valid/labels',
                    'test/images', 'test/labels']

print("Checking dataset structure...")
print("=" * 50)

# Loop through each expected folder
for folder in expected_folders:
    # Check if the folder exists
    if os.path.exists(folder):
        # Count files in the folder
        file_count = len(os.listdir(folder))
        print(f"✅ {folder}: {file_count} files found")
    else:
        print(f"❌ {folder}: NOT FOUND!")

print("=" * 50)
print("\nIf all folders show ✅, you're ready to train!")


## Step 6: Update data.yaml for Colab

This ensures your data.yaml file has the correct paths for Colab.


In [ ]:
# ==================== STEP 6: UPDATE DATA.YAML ====================
# This code updates your data.yaml file to use the correct paths in Colab

# Read the current data.yaml file
# 'r' means read mode - we're only reading, not writing
with open('data.yaml', 'r') as f:
    # f.read() reads the entire file content as a string
    content = f.read()
    print("Current data.yaml content:")
    print(content)
    print("\n" + "=" * 50)

# Update the paths to be absolute paths (full paths from root)
# This ensures YOLO can find your data no matter where it runs
updated_content = """train: train/images
val: valid/images
test: test/images

nc: 3
names: ['glioma', 'meningioma', 'pituitary']
"""

# Write the updated content back to the file
# 'w' means write mode - this will overwrite the existing file
with open('data.yaml', 'w') as f:
    # f.write() writes the string to the file
    f.write(updated_content)

print("✅ data.yaml updated successfully!")
print("\nUpdated content:")
print(updated_content)


## Step 7: Enable GPU (IMPORTANT for Fast Training!)

**Before running training, make sure GPU is enabled:**
1. Click: **Runtime** → **Change runtime type**
2. Select: **GPU** (T4 or better)
3. Click **Save**

This makes training 10x faster!


In [ ]:
# ==================== CHECK GPU AVAILABILITY ====================
# This code checks if GPU is available and enabled

import torch
# torch is PyTorch, the deep learning framework YOLO uses

# Check if CUDA (GPU support) is available
# torch.cuda.is_available() returns True if GPU is available
if torch.cuda.is_available():
    # Get the GPU name
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU is available: {gpu_name}")
    print("You're ready for fast training!")
else:
    print("⚠️ GPU is NOT available. Training will be very slow.")
    print("Please enable GPU: Runtime → Change runtime type → GPU")


## Step 8: Train the Model

**This is the main training step. Training can take 2-6 hours, so be patient!**

⚠️ **Important:**
- Don't close the browser tab during training
- Colab may disconnect after ~90 minutes of inactivity (keep the tab active)
- Training progress will be shown automatically


In [ ]:
# ==================== STEP 7: TRAIN THE MODEL ====================
# This code trains your YOLOv8 model on your brain tumor dataset
# Training can take a long time (hours), so be patient!

# Make sure ultralytics is installed (in case Step 4 was skipped)
import subprocess
import sys
try:
    from ultralytics import YOLO
except ImportError:
    print("Installing ultralytics...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "-q"])
    from ultralytics import YOLO

# YOLO is the main class from ultralytics library
# It handles loading models, training, and predictions

import os
# os is used for file operations

# ==================== CONFIGURATION ====================
# These are settings for your training
# You can change these values to experiment

DATA_YAML = 'data.yaml'  # Path to your dataset configuration file
EPOCHS = 100              # Number of training cycles (how many times the model sees all data)
IMAGE_SIZE = 640         # Size of images in pixels (640x640)
BATCH_SIZE = 16          # Number of images processed at once (higher = faster but needs more memory)

# ==================== VERIFY DATA FILE ====================
# Check if data.yaml exists before starting training
if not os.path.exists(DATA_YAML):
    print(f"ERROR: {DATA_YAML} not found!")
    print("Please make sure data.yaml is in your project folder.")
else:
    print("=" * 50)
    print("BRAIN TUMOR DETECTION - MEDICAL OPTIMIZED TRAINING")
    print("=" * 50)
    
    # ==================== LOAD MODEL ====================
    # Load the pre-trained YOLOv8 model
    # 'yolov8n.pt' is a pre-trained model (n = nano, smallest version)
    # If this file doesn't exist, YOLO will download it automatically
    print("Loading model...")
    model = YOLO('yolov8n.pt')
    # YOLO() creates a YOLO model object
    # The parameter is the path to the pre-trained weights file
    
    print("\nStarting Medical Training...")
    print("Applied Fixes: Disabled Color Jitter (HSV), Lowered Learning Rate, Added Rotation.")
    print("=" * 50)
    
    # ==================== START TRAINING ====================
    try:
        # model.train() starts the training process
        # All the parameters below configure how training happens
        results = model.train(
            data=DATA_YAML,        # Path to dataset configuration
            epochs=EPOCHS,         # How many times to train on all data
            imgsz=IMAGE_SIZE,      # Image size (640x640 pixels)
            batch=BATCH_SIZE,      # Images per batch
            
            # Workers: Number of parallel processes for loading data
            # Colab can handle more workers than local Windows
            workers=4,             # Increased for Colab (was 2 for Windows)
            
            # Patience: Stop training if no improvement for this many epochs
            patience=20,           # Prevents overfitting (training too long)
            
            # Learning rate: How fast the model learns
            lr0=0.001,             # Initial learning rate (start slow for medical images)
            lrf=0.01,              # Final learning rate (end even slower)
            
            # Data augmentation: Modifications to images during training
            # This helps the model learn from variations
            hsv_h=0.0,             # Disable hue change (MRI is grayscale)
            hsv_s=0.0,             # Disable saturation change (MRI is grayscale)
            hsv_v=0.2,             # Keep brightness variation (20% change allowed)
            degrees=15.0,          # Rotate images up to 15 degrees
            translate=0.1,        # Shift images up to 10% horizontally/vertically
            scale=0.2,             # Scale images up to 20% larger/smaller
            fliplr=0.5,            # 50% chance to flip images left-right
            
            # Output settings
            save=True,             # Save the trained model
            plots=True             # Generate training graphs
        )
        
        # ==================== TRAINING COMPLETE ====================
        print("\n" + "=" * 50)
        print("[Training Complete!]")
        print("=" * 50)
        # The best model is saved automatically
        print("Your new best model is at: runs/detect/train/weights/best.pt")
        print("\nYou can download this file to use in your local app!")
        
    except Exception as e:
        # If something goes wrong, this catches the error
        print(f"An error occurred: {e}")
        print("\nCommon issues:")
        print("1. Check if data.yaml paths are correct")
        print("2. Verify all dataset folders exist")
        print("3. Make sure you have enough Colab storage")


## Step 9: Run Testing & Inference on Test Dataset

This step runs inference on your test dataset and saves predictions.



In [ ]:
# ==================== STEP 9: RUN TESTING & INFERENCE ====================
# This code runs inference on your test dataset and saves predictions

from ultralytics import YOLO
import os
from pathlib import Path

# Load the trained model
best_model_path = 'runs/detect/train/weights/best.pt'
test_images_path = 'test/images'

if not os.path.exists(best_model_path):
    print(f"ERROR: Trained model not found at {best_model_path}")
    print("Please run training first.")
else:
    print("Loading trained model...")
    model = YOLO(best_model_path)
    
    # Check if test dataset exists
    if not os.path.exists(test_images_path):
        print(f"⚠️ Test images folder not found at {test_images_path}")
        print("Skipping inference.")
    else:
        print(f"\n✅ Found test images at {test_images_path}")
        num_test_images = len(os.listdir(test_images_path))
        print(f"Running inference on {num_test_images} test images...\n")
        
        # Run inference on test images
        results = model.predict(
            source=test_images_path,
            imgsz=640,
            conf=0.25,
            save=True,           # Save annotated images
            save_txt=True,        # Save predictions as .txt files
            project='runs/detect',
            name='test_results'
        )
        
        print(f"\n✅ Inference complete!")
        print(f"Results saved to: runs/detect/test_results/")
        print(f"Total predictions made: {len(results)}")
        
        # Show summary statistics
        total_detections = sum([len(r.boxes) for r in results])
        print(f"Total detections: {total_detections}")



## Step 10: View Training Results & Metrics

This displays training graphs, metrics, and confusion matrix.



In [ ]:
# ==================== STEP 10: VIEW RESULTS & METRICS ====================
# This displays training results, metrics, and test predictions

import os
import glob
from PIL import Image
import matplotlib.pyplot as plt

print("=" * 60)
print("TRAINING & TESTING RESULTS SUMMARY")
print("=" * 60)

# Show training results location
train_results_dir = 'runs/detect/train'
if os.path.exists(train_results_dir):
    print(f"\n✅ Training results saved at: {train_results_dir}/")
    
    # List result files
    results_files = glob.glob(os.path.join(train_results_dir, '*.png'))
    if results_files:
        print("\nTraining graphs found:")
        for f in sorted(results_files):
            print(f"  - {os.path.basename(f)}")
    
    # Show metrics if available
    metrics_file = os.path.join(train_results_dir, 'results.csv')
    if os.path.exists(metrics_file):
        import pandas as pd
        df = pd.read_csv(metrics_file)
        print("\n📊 Final Training Metrics (Last Epoch):")
        print(df.tail(1).to_string())

# Show test results
test_results_dir = 'runs/detect/test_results'
if os.path.exists(test_results_dir):
    test_images = glob.glob(os.path.join(test_results_dir, '*.jpg')) + \
                   glob.glob(os.path.join(test_results_dir, '*.png'))
    print(f"\n✅ Test predictions saved at: {test_results_dir}/")
    print(f"   Total annotated test images: {len(test_images)}")

# Display sample training graphs
print("\n" + "=" * 60)
print("DISPLAYING SAMPLE RESULTS")
print("=" * 60)

sample_graphs = glob.glob(os.path.join(train_results_dir, 'results.png')) + \
                glob.glob(os.path.join(train_results_dir, 'confusion_matrix*.png'))

if sample_graphs:
    for graph_path in sample_graphs[:3]:  # Show first 3 graphs
        try:
            img = Image.open(graph_path)
            plt.figure(figsize=(12, 5))
            plt.imshow(img)
            plt.axis('off')
            plt.title(os.path.basename(graph_path))
            plt.tight_layout()
            plt.show()
            print(f"✅ Displayed: {os.path.basename(graph_path)}")
        except Exception as e:
            print(f"Could not display {graph_path}: {e}")
else:
    print("No result graphs found yet. Run training first.")



## Step 11: Run Streamlit Web App in Colab

This runs your Streamlit app in Colab using ngrok for public access.



In [ ]:
# ==================== STEP 11: RUN STREAMLIT WEB APP IN COLAB ====================
# This runs your Streamlit app and creates a public URL via ngrok

import subprocess
import time

print("=" * 60)
print("LAUNCHING STREAMLIT WEB APP IN COLAB")
print("=" * 60)

# Install required packages
print("\n1️⃣ Installing required packages...")
!pip install -q streamlit pyngrok

# Check if app.py exists
app_path = 'app.py'
if not os.path.exists(app_path):
    print(f"\n⚠️ {app_path} not found in current directory!")
    print("Make sure app.py is in your project root.")
    print("\nAlternatively, copy this streamlit app template to create a minimal app.py:")
    print("""
# Example app.py for Colab
import streamlit as st
from PIL import Image
import os

st.title("🧠 Brain Tumor Detection")

uploaded_file = st.file_uploader("Upload an MRI scan image", type=['jpg','png','jpeg'])

if uploaded_file:
    image = Image.open(uploaded_file)
    st.image(image, caption="Uploaded Image")
    st.info("✅ Image uploaded successfully!")
    """)
else:
    print(f"\n✅ Found {app_path}")
    print("\n2️⃣ Starting Streamlit app...\n")
    
    # Start Streamlit in background using python -m
    subprocess.Popen([
        'python', '-m', 'streamlit', 'run', app_path,
        '--server.port=8501',
        '--server.headless=true',
        '--logger.level=error'
    ])
    
    time.sleep(3)
    
    # Setup public URL
    print("3️⃣ Setting up public URL via tunneling...\n")
    
    # Try method 1: ngrok (requires auth token)
    try:
        from pyngrok import ngrok
        
        # Check if auth token is set in environment or config
        # Users can set ngrok auth token by creating a ~/.ngrok2/ngrok.yml file
        # or by signing up at https://ngrok.com and using ngrok config add-authtoken
        
        try:
            public_url = ngrok.connect(8501)
            print("=" * 60)
            print("🎉 STREAMLIT APP IS RUNNING!")
            print("=" * 60)
            print(f"\n📱 Public URL: {public_url}")
            print("\nYou can now:")
            print("  1. Click the link above to open your web app")
            print("  2. Upload MRI images for brain tumor detection")
            print("  3. View predictions in real-time")
            print("\n⏰ The URL is active as long as this cell is running.")
            print("   Keep this Colab tab open to use the app.")
            print("\n" + "=" * 60)
            
            # Keep the tunnel alive
            ngrok_process = ngrok.get_ngrok_process()
            ngrok_process.proc.wait()
            
        except Exception as ngrok_error:
            print(f"⚠️ ngrok tunnel failed (requires auth token): {str(ngrok_error)[:100]}")
            print("\n" + "=" * 60)
            print("🎉 STREAMLIT APP IS RUNNING (LOCAL MODE)")
            print("=" * 60)
            print("\n📝 Your app is running, but ngrok requires authentication.")
            print("\nOptions to access your app:")
            print("\n  Option 1: Use ngrok auth token (Recommended)")
            print("    - Sign up: https://dashboard.ngrok.com/signup")
            print("    - Get token: https://dashboard.ngrok.com/get-started/your-authtoken")
            print("    - Add this to the next cell:")
            print("      from pyngrok import ngrok")
            print("      ngrok.set_auth_token('YOUR_TOKEN_HERE')")
            print("\n  Option 2: Access via Colab's built-in browser")
            print("    - The app is running at: http://localhost:8501/")
            print("    - Streamlit may detect Colab and work automatically")
            print("\n  Option 3: Use alternative tunneling service")
            print("    - Install: !pip install cloudflare-py")
            print("    - Or use: !pip install localtunnel")
            print("\n" + "=" * 60)
            
            # Keep streamlit running
            while True:
                time.sleep(1)
    
    except ImportError:
        print("pyngrok not available. Starting Streamlit in local mode.\n")
        print("=" * 60)
        print("🎉 STREAMLIT APP IS RUNNING")
        print("=" * 60)
        print("\nYour app is available at: http://localhost:8501/")
        print("\nTo access from outside Colab, install an auth token:")
        print("  1. Sign up at: https://dashboard.ngrok.com/signup")
        print("  2. Get your token from: https://dashboard.ngrok.com/get-started/your-authtoken")
        print("  3. Add to a new cell:")
        print("     from pyngrok import ngrok")
        print("     ngrok.set_auth_token('YOUR_TOKEN')")
        print("\n" + "=" * 60)
        
        # Keep streamlit running
        while True:
            time.sleep(1)



## Step 11b: Set Up ngrok Auth Token (For Public URL Access)

If you want to access your Streamlit app from outside Colab with a public URL, follow these steps:

1. **Sign up for ngrok** (free): https://dashboard.ngrok.com/signup
2. **Get your auth token**: https://dashboard.ngrok.com/get-started/your-authtoken
3. **Copy your token** from that page
4. **Run the cell below** and paste your token when prompted
5. **Re-run Step 11** to get a public URL



In [ ]:
# ==================== STEP 11B: SET NGROK AUTH TOKEN ====================
# This code sets up your ngrok auth token for public URL access

from pyngrok import ngrok
import getpass

print("=" * 60)
print("NGROK AUTH TOKEN SETUP")
print("=" * 60)

# Get the auth token from user
print("\n📋 Steps:")
print("1. Go to: https://dashboard.ngrok.com/get-started/your-authtoken")
print("2. Copy your auth token (it looks like: 2a3_5x... )")
print("3. Paste it below when prompted\n")

# Prompt for token (getpass hides the input for security)
auth_token = getpass.getpass("Enter your ngrok auth token: ").strip()

if auth_token:
    try:
        # Set the auth token
        ngrok.set_auth_token(auth_token)
        print("\n✅ Auth token set successfully!")
        print("\n📝 Next steps:")
        print("1. Scroll back up to Step 11")
        print("2. Click the Run button (▶️) to re-run that cell")
        print("3. You should now get a public URL like: https://xxxx-xx-xxx-xxx.ngrok.io")
        print("\n" + "=" * 60)
    except Exception as e:
        print(f"\n❌ Error setting token: {e}")
        print("Please check your token and try again.")
else:
    print("\n⚠️ No token provided. Skipping setup.")


In [ ]:
# ==================== DOWNLOAD TRAINED MODEL ====================
# This code helps you download your trained model file

from google.colab import files
# files is a Colab utility for downloading files

# Check if the best model exists
model_path = 'runs/detect/train/weights/best.pt'
if os.path.exists(model_path):
    print("Downloading best.pt...")
    # files.download() downloads a file to your computer
    files.download(model_path)
    print("✅ Download complete!")
    print("\nYou can now use this model in your local app.py")
else:
    print("❌ Model file not found. Training may not have completed.")
    print("Check the path: runs/detect/train/weights/best.pt")


### Next Steps:
1. Download the `best.pt` file (from Step 9)
2. Place it in your local project folder
3. Run your `app.py` locally with the new model

### Tips:
- The model is saved in `runs/detect/train/weights/best.pt`
- Training graphs are in `runs/detect/train/`
- You can re-run training with different settings to experiment

---

**Need help?** Check the `COLAB_SETUP_GUIDE.md` file for detailed troubleshooting!
